# Aarogya — Extensive Fine-tuning of Gemma 4 / 3 (Unsloth + QLoRA)

**Hackathon:** Gemma 4 Good · Special Track: **Unsloth** ($10K)

**Hardware:** Kaggle GPU **T4 × 2** · Internet **ON**

**Runtime:** ~2-3 hours for 1500 steps on 5000 samples

Auto-falls-back to Gemma 3 4B if Unsloth hasn't published a Gemma 4 4B build yet.

## 1. Install Unsloth (uses prebuilt wheels — no source compilation)

In [ ]:
%%capture
# Unsloth official Kaggle T4 install (prebuilt wheels — no compilation)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets  # for multi-source dataset loading


In [ ]:
# Sanity check — fail fast if Unsloth did not install
from unsloth import FastLanguageModel
import torch
print(f"Unsloth OK · GPUs: {torch.cuda.device_count()} · CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}")


## 2. Clone the Aarogya repo + build dataset

In [ ]:
import os
if not os.path.exists("/kaggle/working/aarogya"):
    !git clone https://github.com/shabdkumar3/aarogya.git /kaggle/working/aarogya
else:
    !cd /kaggle/working/aarogya && git pull

%cd /kaggle/working/aarogya/finetuning

# Build extensive multi-source dataset (~15K train + 2K val)
!python prepare_dataset.py
!echo "Train lines:   Val lines: "


## 3. Pick the best Gemma model available on Unsloth (Gemma 4 → Gemma 3 fallback)

In [ ]:
from huggingface_hub import HfApi

CANDIDATES = [
    "unsloth/gemma-4-4b-it-unsloth-bnb-4bit",
    "unsloth/gemma-4-4b-it",
    "unsloth/gemma-3-4b-it-bnb-4bit",
    "unsloth/gemma-3-4b-it",
]
api = HfApi()
MODEL_NAME = None
for m in CANDIDATES:
    try:
        api.repo_info(m)
        MODEL_NAME = m
        print(f"Using model: {MODEL_NAME}")
        break
    except Exception:
        print(f"Not found: {m}")

assert MODEL_NAME, "No Gemma model found on HuggingFace!"


## 4. Load model with QLoRA (4-bit)

In [ ]:
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    target_modules=["q_proj","k_proj","v_proj","o_proj",
                     "gate_proj","up_proj","down_proj"],
)
print(model.print_trainable_parameters())


## 5. Train (1500 steps · ~2-3h on T4×2)

In [ ]:
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

train_ds = load_dataset("json", data_files="train_data.jsonl", split="train")
val_ds   = load_dataset("json", data_files="val_data.jsonl",   split="train")
print(f"Train: {len(train_ds)}  Val: {len(val_ds)}")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,    # effective batch = 16
        warmup_steps=100,
        max_steps=2000,                   # ~3-4h on T4x2 with 15K samples
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=50,
        evaluation_strategy="steps",
        eval_steps=200,
        save_strategy="steps",
        save_steps=500,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="/kaggle/working/checkpoints",
        report_to="none",
    ),
)

print("Starting training — 2000 steps, ~3-4 hours on T4x2...")
trainer_stats = trainer.train()
print(f"Training complete! Loss: {trainer_stats.training_loss:.4f}")


## 6. Save LoRA adapter

In [ ]:
model.save_pretrained("/kaggle/working/aarogya-lora")
tokenizer.save_pretrained("/kaggle/working/aarogya-lora")
!ls -lh /kaggle/working/aarogya-lora
print("LoRA adapter saved!")


## 7. Export merged GGUF for Ollama (optional but nice for Ollama track)

In [ ]:
try:
    model.save_pretrained_gguf(
        "/kaggle/working/aarogya-gguf",
        tokenizer,
        quantization_method="q4_k_m",
    )
    !ls -lh /kaggle/working/aarogya-gguf
    print("GGUF saved!")
except Exception as e:
    print(f"GGUF export skipped (optional): {e}")


## 8. Sample inference — fine-tuned vs base (for the writeup)

In [ ]:
FastLanguageModel.for_inference(model)

test_cases = [
    "Bachcha 2 din se paani-wala dast, kuch khaya nahi, aankhein andar dhans gayi",
    "Patient has fever 102F for 3 days with severe headache and stiff neck",
    "Aurat 8 mahine pregnant, haath-paon mein sujan, sar dard aur aankhon ke aage andhera",
    "Child 3 years, arm very thin, MUAC 10.5cm, not eating for days",
]

SYSTEM = """You are Aarogya, a medical triage assistant for ASHA workers. Output ONLY valid JSON triage."""

for case in test_cases:
    prompt = f"<start_of_turn>user
{SYSTEM}

Symptoms: {case}<end_of_turn>
<start_of_turn>model
"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=256, temperature=0.1, do_sample=True)
    response = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    print(f"
Case: {case[:60]}...")
    print(f"Response: {response[:400]}")
    print("-" * 60)
